In [10]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [11]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

/Users/abdulhaseeb/Desktop/mlProject1/Regression_ML_EndtoEnd/.venv/bin/python
3.0.4
/Users/abdulhaseeb/Desktop/mlProject1/Regression_ML_EndtoEnd/.venv/lib/python3.11/site-packages/xgboost/__init__.py


In [12]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

In [13]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv("/Users/abdulhaseeb/Desktop/mlProject1/Regression_ML_EndtoEnd/data/processed/feature_engineered_train.csv")
eval_df  = pd.read_csv("/Users/abdulhaseeb/Desktop/mlProject1/Regression_ML_EndtoEnd/data/processed/feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [14]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [15]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
mlflow.set_tracking_uri("/Users/abdulhaseeb/Desktop/mlProject1/Regression_ML_EndtoEnd/mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

2026/07/02 01:13:16 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-07-02 01:13:16,978] A new study created in memory with name: no-name-424a0a2e-281b-4e8d-b7e3-d827730d8da4
[I 2026-07-02 01:13:21,342] Trial 0 finished with value: 87110.50904486078 and parameters: {'n_estimators': 295, 'max_depth': 5, 'learning_rate': 0.011994719029347135, 'subsample': 0.629212483796678, 'colsample_bytree': 0.9305213791888759, 'min_child_weight': 6, 'gamma': 4.304968896476468, 'reg_alpha': 1.445929311297854e-05, 'reg_lambda': 1.1683585197365513e-07}. Best is trial 0 with value: 87110.50904486078.
[I 2026-07-02 01:13:42,285] Trial 1 finished with value: 68540.41551262558 and parameters: {'n_estimators': 802, 'max_depth': 10, 'learning_rate': 0.026507414055411214, 'subsample': 0.5893394227049227, 'colsample_bytree': 0.5600768543406378, 'min_child_weight': 10, 'gamma': 1.412190514586058, 'reg_alpha': 0.7683106603930949, 'reg_lam

Best params: {'n_estimators': 802, 'max_depth': 10, 'learning_rate': 0.026507414055411214, 'subsample': 0.5893394227049227, 'colsample_bytree': 0.5600768543406378, 'min_child_weight': 10, 'gamma': 1.412190514586058, 'reg_alpha': 0.7683106603930949, 'reg_lambda': 0.266225942180956}


In [16]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 30875.159107865264
RMSE: 69592.70174711428
R²: 0.9625727924600473


/Users/abdulhaseeb/Desktop/mlProject1/Regression_ML_EndtoEnd/.venv/lib/python3.11/site-packages/xgboost/sklearn.py:1028: UserWarning: [01:16:49] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/07/02 01:16:52 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/07/02 01:16:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
